# V1.4 模型适用性与传播尺度标定快速复现

本 Notebook 仅演示波长尺度、归一化标量复场和降阶传播模型。所有量均不对应绝对源功率、现实器件阈值、毁伤概率或作用距离。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from hpm_platform.ui.project_model import CAEProject
from hpm_platform.validation.model_validity import assess_model_validity
from hpm_platform.validation.backend_calibration import calibrate_backend_scales
project = CAEProject.load_yaml(ROOT / "configs" / "cae_project_v14.yaml")
project.meta.name, project.schema_version

## 四种传播后端的适用性诊断

In [ ]:
backends = ["free_space_green", "image_ray", "aperture_cavity_rom", "hybrid_scene"]
rows = []
for backend in backends:
    report = assess_model_validity(project, backend)
    rows.append({"传播后端": report.backend_name, "适用性得分": report.score, "结论": report.level, "检查项数": len(report.checks)})
pd.DataFrame(rows)

## 含 0.25% 归一化复场噪声的尺度参数标定

In [ ]:
result = calibrate_backend_scales(
    project,
    reference_backend="hybrid_scene",
    candidate_backend="hybrid_scene",
    reference_scales=(0.86, 0.72, 0.93),
    initial_scales=(0.50, 0.40, 0.40),
    samples_per_axis=21,
    noise_std_fraction=0.0025,
    maximum_evaluations=60,
)
pd.DataFrame({
    "参数": ["直达尺度", "反射尺度", "腔体尺度"],
    "参考值": [0.86, 0.72, 0.93],
    "初始值": result.initial_scales,
    "标定值": result.fitted_scales,
})

In [ ]:
pd.Series(result.summary_dict(), name="结果")

## 解读

标定误差很低只说明在当前合成参考模型、采样布局和噪声水平下，三个尺度具有较好的数值可辨识性。正式论文必须增加独立验证点、模型失配、参数相关性和置信区间分析。